## 1. Setup

Basic EDA, inspecting raw block (the latest), raw transaction within a block, and 100 contiguous blocks.
We first write a simple rpc to retreive the jsons.

In [2]:
import os, json, time, requests, pandas as pd
from web3 import Web3

URL = f"https://eth-mainnet.g.alchemy.com/v2/{os.getenv('ALCHEMY_KEY')}"

# RPC 2.0
# jsonrpc.org/specification
def rpc(method, params):
    r = requests.post(URL, json={"jsonrpc":"2.0","id":1,"method":method,"params":params})
    j = r.json()
    if "error" in j:
        raise RuntimeError(j["error"])
    return j["result"]

### 1-1. Raw Block

In [ ]:
# https://ethereum.org/developers/docs/apis/json-rpc/#eth-blocknumber
# https://ethereum.org/developers/docs/apis/json-rpc/#eth-getblockbynumber
latest = int(rpc("eth_blockNumber", []), 16)                # eth_blockNumber returns hex string
block = rpc("eth_getBlockByNumber", [hex(latest), True])    # eth-getblockbynumber 
print(json.dumps(block, indent=2)[:3000]) 


{
  "hash": "0x19930e1087a2a25aeda25e2dfa63335b90d9340b67647805a6b07d3d3cc55d6b",
  "parentHash": "0xf5f538fec1725865a02003c5ede513652771eab31089a26bf481d991327ef645",
  "sha3Uncles": "0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a7413f0a142fd40d49347",
  "miner": "0xdadb0d80178819f2319190d340ce9a924f783711",
  "stateRoot": "0x4ad5251876d5df5899373d34a3aaaa75e1e8ec69aa6ce51e73c27a559bab5962",
  "transactionsRoot": "0x4dc1b7e7d20e100e1f531968f055ee8dc9fa0f127de295d0025560698d03abe2",
  "receiptsRoot": "0xce9276c2f96264b74e5ec8817bf3509d55dd5ab5986d0ca399149fa81475c430",
  "logsBloom": "0x6fe5087ffbef7f3ff6ff37ffe7dde5f1ffd9d7ffffb4e73ffde1f7d9ffb4ddffa7fecf36e02f83ae97d17bfff9bf6dd5f7d196da8cef6e77bfffdff7fdbef17dfff7f5edf5ff1b3e6b56d929eefdaffebfdef47cd7fc3c7d55df5ffed97d69c8d647f8eedbffc454d15ef7ff1fe76efbb4fbf77e237eae7c13b773d7339d8beffeb7aeff1fa2a79fbe6fd6ff5e7e5763d59351777ff7fbcdfeebdde2d7bfeffc1edc17fbf7fdfb5ff7797ffcf9c99f75e575c9effeefe57f3b74ffea3e5e5bfeffbfdffee7e59ee9e6ef1f

### 1-2. Raw Transaction

In [ ]:
tx = block["transactions"][0]
print(json.dumps(tx, indent=2)) 

{
  "type": "0x2",
  "chainId": "0x1",
  "nonce": "0x3a",
  "gas": "0x186a0",
  "maxFeePerGas": "0x136035290",
  "maxPriorityFeePerGas": "0x1dcd6500",
  "to": "0x999b49c0d1612e619a4a4f6280733184da025108",
  "value": "0x0",
  "accessList": [],
  "input": "0x095ea7b30000000000000000000000004313c378cc91ea583c91387b9216e2c03096b27f000000000000000000000000000000000000000000000073d4051301d2145ee2",
  "r": "0xb707fe8e2c84d367b300607118161a55240e2e9b1848e098975c96c1b8b658b",
  "s": "0x4d03d3523b3decca607e3dc06a64f6122fada92226b613de1a407c70c8cceaf4",
  "yParity": "0x1",
  "v": "0x1",
  "hash": "0xc245864786c73c1f60dff4ac5d1d9bf6eb9504d481798d6010f8f2034a86c797",
  "blockHash": "0x19930e1087a2a25aeda25e2dfa63335b90d9340b67647805a6b07d3d3cc55d6b",
  "blockNumber": "0x17e2007",
  "transactionIndex": "0x0",
  "from": "0x1da8498facdbb3089197bf12b9f1d24c04c21922",
  "gasPrice": "0xcd3d8aa3"
}


### 1-3. 100 Blocks

In [ ]:
# Pull 100 blocks, flatten to DataFrame
rows = []
start = latest - 99
for n in range(start, latest + 1):
    b = rpc("eth_getBlockByNumber", [hex(n), True])
    with open(f"../data/raw/block_{n}.json", "w") as f:
        json.dump(b, f)
    ts = int(b["timestamp"], 16)
    for tx in b["transactions"]:
        rows.append({
            "block": n,
            "ts": ts,
            "from": tx["from"],
            "to": tx["to"],
            "value_wei": int(tx["value"], 16),
            "gas": int(tx["gas"], 16),
            "input": tx["input"],
        })
    time.sleep(0.05)
    if n % 10 == 0:
        print(f"pulled block {n}")

df = pd.DataFrame(rows)
df["value_eth"] = df["value_wei"] / 1e18
df["is_contract_call"] = df["input"] != "0x"
print(f"\nTotal txs: {len(df)}")
print(f"Unique senders: {df['from'].nunique()}")
print(f"Contract call ratio: {df['is_contract_call'].mean():.2%}")
df.head()

pulled block 25042860
pulled block 25042870
pulled block 25042880
pulled block 25042890
pulled block 25042900
pulled block 25042910
pulled block 25042920
pulled block 25042930
pulled block 25042940
pulled block 25042950

Total txs: 32808
Unique senders: 18683
Contract call ratio: 57.43%


,block,ts,from,to,value_wei,gas,input,value_eth,is_contract_call
0,25042852,1778153195,0x5875db54cd9ae2b2a875e09bb731772297ae9d92,0x0000000aa232009084bd71a5797d089aa4edfad4,0,100000,0x1828e0e70000000000000000000000005875db54cd9a...,0.0,True
1,25042852,1778153195,0xfd8c07484662cdfa25d8c61190208313f8a84519,0x80a64c6d7f12c47b7c66c5b4e20e72bc1fcd5d9e,0,296679,0x3d0e3ec5000000000000000000000000000000000000...,0.0,True
2,25042852,1778153195,0x7934512c688e6e97b5038c271b47c6d7a0db54c4,0x80a64c6d7f12c47b7c66c5b4e20e72bc1fcd5d9e,0,296679,0x3d0e3ec5000000000000000000000000000000000000...,0.0,True
3,25042852,1778153195,0xae2fc483527b8ef99eb5d9b44875f005ba1fae13,0x1f2f10d1c40777ae1da742455c65828ff36df387,12077291,129317,0x041ea77062de531a98c46ecfb78a5cf7153b5da9527c...,0.0,True
4,25042852,1778153195,0x150363ec6dcc84b6c7d6d6283fa3746176f72065,0x0000000000001ff3684f28c67538d4d072c22734,10434398622881295,190659,0x2213bc0b0000000000000000000000007f54f05635d1...,0.010434,True


### 1-4. ERC-20 Transfer Logs from 100 Blocks

In [ ]:
# ERC-20 Transfer logs for 100 blocks
# "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"
TRANSFER = Web3.keccak(text="Transfer(address,address,uint256)").hex()
all_logs = []
for start in range(latest-99, latest+1, 10):
    end = min(start+9, latest)
    chunk = rpc("eth_getLogs", [{
        "fromBlock": hex(start),
        "toBlock":   hex(end),
        "topics":    [TRANSFER],
    }])
    all_logs.extend(chunk)
    time.sleep(0.1)

print(json.dumps(all_logs[0], indent=2)) 

{
  "address": "0x794269de14d78d5dc4e77da55d69e811054b1110",
  "topics": [
    "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef",
    "0x000000000000000000000000d9f8bbe437a3423b725c6616c1b543775ecf1110",
    "0x0000000000000000000000009966880abe51279f66c52e59ac3583c360bf3090"
  ],
  "data": "0x000000000000000000000000000000000000000000194512dfee947e840ada4b",
  "blockHash": "0x1889b3a16166570fe192454fa4f692f0482f6b55ecbf358f17cd0a8c335ad830",
  "blockNumber": "0x17e1f11",
  "blockTimestamp": "0x69fc6ffb",
  "transactionHash": "0x3da8ef5e5458c235925933a613b3c7d8fb1d8385b39f57597a626169a6773fc0",
  "transactionIndex": "0x0",
  "logIndex": "0x0",
  "removed": false
}
